In [84]:
import pandas as pd

# define list of crops to process
crops = ["sugarcane", "arhar", "rice", "gram", "pulses"]  # add your crops here

# process each crop
for crop in crops:
    # load dataset
    crop_raw = pd.read_csv(f"{crop}_raw.csv")
    
    # identify columns
    area_cols = [c for c in crop_raw.columns if "Area (Hectare)" in c]
    prod_cols = [c for c in crop_raw.columns if "Production (Tonnes)" in c]
    yield_cols = [c for c in crop_raw.columns if "Yield (Tonne/Hectare)" in c]
    
    # combine them
    # Convert area columns to numeric, removing commas
    for col in area_cols:
        crop_raw[col] = pd.to_numeric(crop_raw[col].astype(str).str.replace(',', ''), errors='coerce')
    
    crop_raw[f"{crop}_area"] = crop_raw[area_cols].sum(axis=1, skipna=True)
    # Convert production columns to numeric, removing commas
    for col in prod_cols:
        crop_raw[col] = pd.to_numeric(crop_raw[col].astype(str).str.replace(',', ''), errors='coerce')
    
    crop_raw[f"{crop}_production"] = crop_raw[prod_cols].sum(axis=1, skipna=True)
    crop_raw[f"{crop}_yield"] = crop_raw[yield_cols].sum(axis=1, skipna=True)
    
    # keep only required columns
    crop_clean = crop_raw[["State", "District", "Year",
                           f"{crop}_area", f"{crop}_production", f"{crop}_yield"]]
    # store each crop dataframe dynamically (e.g., sugarcane_df, rice_df, ...)
    globals()[f"{crop}_df"] = crop_clean
    
    print(f"\n{crop.upper()} Data:")
    print(crop_clean.head())
    
    # optionally save to CSV
    crop_clean.to_csv(f"{crop}_clean.csv", index=False)
    
# load dataset
cotton_raw = pd.read_csv("cotton_raw.csv")

# identify columns
area_cols = [c for c in cotton_raw.columns if "Area (Hectare)" in c]
prod_cols = [c for c in cotton_raw.columns if "Production (Bales)" in c]
yield_cols = [c for c in cotton_raw.columns if "Yield (Bales/Hectare)" in c]

# combine them
cotton_raw["cotton_area"] = cotton_raw[area_cols].sum(axis=1, skipna=True)
cotton_raw["cotton_production"] = cotton_raw[prod_cols].sum(axis=1, skipna=True)
cotton_raw["cotton_yield"] = cotton_raw[yield_cols].sum(axis=1, skipna=True)

# keep only required columns
cotton_df = cotton_raw[["State", "District", "Year",
                           "cotton_area", "cotton_production", "cotton_yield"]]

print(cotton_df.head())
cotton_df.to_csv("cotton_clean.csv", index=False)


SUGARCANE Data:
                            State     District         Year  sugarcane_area  \
0  1. Andaman and Nicobar Islands  1. Nicobars  2013 - 2014           11.00   
1                             NaN          NaN  2014 - 2015           22.20   
2                             NaN          NaN  2015 - 2016            6.00   
3                             NaN          NaN  2016 - 2017           18.43   
4                             NaN          NaN  2017 - 2018           17.00   

   sugarcane_production  sugarcane_yield  
0                 373.4            33.95  
1                 254.0            11.44  
2                  12.0             2.00  
3                  27.0             1.47  
4                  13.2             0.78  

ARHAR Data:
                            State                     District         Year  \
0  1. Andaman and Nicobar Islands  1. North and middle andaman  2013 - 2014   
1                             NaN                          NaN  2014 - 2015   


### Some more cleaning

In [85]:
def clean_state_district(data):
    data["State"] = data["State"].ffill()
    data["District"] = data["District"].ffill()
    data["District"] = data["District"].str.replace(r"^\d+\.\s*", "", regex=True)
    data["State"] = data["State"].str.replace(r"^\d+\.\s*", "", regex=True)
    
     # clean year
    data["Year"] = data["Year"].str.replace(" ", "")
    
    # rename columns
    data = data.rename(columns={
        f"{crop}_area": f"{crop}_area",
        f"{crop}_production": f"{crop}_production",
        f"{crop}_yield": f"{crop}_yield"
    })
    
    # keep only needed columns
    data = data[["State","District","Year",
             f"{crop}_area",
             f"{crop}_production",
             f"{crop}_yield"]]
    
    return data

# use a dictionary (not a set) to store dataframes by crop name
clean_dfs = {}

for name in crops + ["cotton"]:
    file_path = f"{name}_clean.csv"
    temp = pd.read_csv(file_path)
    crop = name
    temp = clean_state_district(temp)
    temp.to_csv(file_path, index=False)
    clean_dfs[name] = temp

# keep commonly used variables in notebook
cotton_clean = clean_dfs["cotton"]
for name in crops:
    globals()[f"{name}_clean"] = clean_dfs[name]

### Merge Data into single dataframe

In [86]:
from functools import reduce

dfs = [rice_clean, cotton_clean, arhar_clean, sugarcane_clean, gram_clean, pulses_clean]

agri_df = reduce(
    lambda left, right: pd.merge(
        left, right,
        on=["State","District","Year"],
        how="outer"
    ),
    dfs
)

# Convert production columns to numeric, handling any string values with commas
for col in ["rice_production", "cotton_production", "arhar_production", "sugarcane_production", "gram_production", "pulses_production"]:
    agri_df[col] = pd.to_numeric(agri_df[col].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
    
# Convert area columns to numeric, handling any string values with commas
for col in ["rice_area", "cotton_area", "arhar_area", "sugarcane_area", "gram_area", "pulses_area"]:
    agri_df[col] = pd.to_numeric(agri_df[col].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

agri_df = agri_df.fillna(0)

agri_df.head(11)

,State,District,Year,rice_area,rice_production,rice_yield,cotton_area,cotton_production,cotton_yield,arhar_area,...,arhar_yield,sugarcane_area,sugarcane_production,sugarcane_yield,gram_area,gram_production,gram_yield,pulses_area,pulses_production,pulses_yield
0,Andaman and Nicobar Islands,Nicobars,2013-2014,2.65,6.30,2.38,0.0,0.0,0.0,0.0,...,0.0,11.00,373.4,33.95,0.0,0.0,0.0,0.0,0.0,0.0
1,Andaman and Nicobar Islands,Nicobars,2014-2015,4.60,10.80,2.35,0.0,0.0,0.0,0.0,...,0.0,22.20,254.0,11.44,0.0,0.0,0.0,0.0,0.0,0.0
2,Andaman and Nicobar Islands,Nicobars,2015-2016,5.00,11.20,2.24,0.0,0.0,0.0,0.0,...,0.0,6.00,12.0,2.00,0.0,0.0,0.0,0.0,0.0,0.0
3,Andaman and Nicobar Islands,Nicobars,2016-2017,6.67,9.54,1.43,0.0,0.0,0.0,0.0,...,0.0,18.43,27.0,1.47,0.0,0.0,0.0,0.0,0.0,0.0
4,Andaman and Nicobar Islands,Nicobars,2017-2018,1.00,0.53,0.53,0.0,0.0,0.0,0.0,...,0.0,17.00,13.2,0.78,0.0,0.0,0.0,0.0,0.0,0.0
5,Andaman and Nicobar Islands,Nicobars,2018-2019,4.10,6.01,1.47,0.0,0.0,0.0,0.0,...,0.0,3.00,10.0,3.33,0.0,0.0,0.0,0.0,0.0,0.0
6,Andaman and Nicobar Islands,Nicobars,2019-2020,3.10,4.30,1.39,0.0,0.0,0.0,0.0,...,0.0,3.70,15.0,4.05,0.0,0.0,0.0,0.0,0.0,0.0
7,Andaman and Nicobar Islands,Nicobars,2020-2021,0.00,0.00,0.00,0.0,0.0,0.0,0.0,...,0.0,3.30,13.5,4.09,0.0,0.0,0.0,0.0,0.0,0.0
8,Andaman and Nicobar Islands,Nicobars,2021-2022,0.00,0.00,0.00,0.0,0.0,0.0,0.0,...,0.0,5.50,30.1,5.47,0.0,0.0,0.0,0.0,0.0,0.0
9,Andaman and Nicobar Islands,Nicobars,2022-2023,1.00,1.26,1.26,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0


In [87]:
agri_df["total_area"] = (
      agri_df["rice_area"]
    + agri_df["cotton_area"]
    + agri_df["arhar_area"]
    + agri_df["sugarcane_area"]
)


agri_df["total_production"] = (
      agri_df["rice_production"]
    + agri_df["cotton_production"]
    + agri_df["arhar_production"]
    + agri_df["sugarcane_production"]
)

In [88]:
agri_final = agri_df

### Remove districts having Missing Values

In [89]:
agri_final["year_num"] = agri_final["Year"].str[:4].astype(int)

required_years = set(range(2014, 2023))

district_years = (
    agri_final.groupby(["State","District"])["year_num"]
    .apply(set)
)

valid_districts = district_years[
    district_years.apply(lambda x: required_years.issubset(x))
].index

agri_final = agri_final.set_index(["State","District"])

agri_final = agri_final.loc[valid_districts].reset_index()

### Add Rows for 2023 and 2024

In [90]:
future_years = [2023, 2024, 2025]

new_rows = []

for (state, district), group in agri_final.groupby(["State","District"]):

    for y in future_years:
        row = {
            "State": state,
            "District": district,
            "year_num": y
        }
        new_rows.append(row)

future_df = pd.DataFrame(new_rows)

agri_final = pd.concat([agri_final, future_df], ignore_index=True)

In [91]:
agri_final = agri_final.sort_values(
    ["State","District","year_num"]
)

# from the df only keep required columns: state, district, year, total_area, total_production.
agri_final = agri_final[["State", "District", "Year", "total_area", "total_production"]]

agri_final.head(5)

,State,District,Year,total_area,total_production
0,Andaman and Nicobar Islands,Nicobars,2013-2014,13.65,379.70
1,Andaman and Nicobar Islands,Nicobars,2014-2015,26.80,264.80
2,Andaman and Nicobar Islands,Nicobars,2015-2016,11.00,23.20
3,Andaman and Nicobar Islands,Nicobars,2016-2017,25.10,36.54
4,Andaman and Nicobar Islands,Nicobars,2017-2018,18.00,13.73


### Interpolate the Variables

In [95]:
crop_vars = ["total_area", "total_production"]

# Ensure year_num exists
if "year_num" not in agri_final.columns:
    agri_final["year_num"] = pd.to_numeric(
        agri_final["Year"].astype(str).str[:4],
        errors="coerce"
    )

# Fill missing year_num for future rows
missing_year = agri_final["year_num"].isna()
if missing_year.any():
    agri_final.loc[missing_year, "year_num"] = (
        agri_final.loc[missing_year]
        .groupby(["State", "District"])
        .cumcount() + 2023
    )

agri_final["year_num"] = agri_final["year_num"].astype("Int64")

# Sort by State, District, and year_num
agri_final = agri_final.sort_values(["State", "District", "year_num"]).reset_index(drop=True)

# Interpolate for each district
for var in crop_vars:
    # Convert to numeric
    agri_final[var] = pd.to_numeric(agri_final[var], errors="coerce")
    
    # Interpolate within each group
    agri_final[var] = (
        agri_final.groupby(["State", "District"])[var]
        .transform(lambda x: x.interpolate(method="linear", limit_direction="forward"))
    )

# Fill any remaining NaN with 0
agri_final[crop_vars] = agri_final[crop_vars].fillna(0)

# Rebuild Year column
agri_final["Year"] = (
    agri_final["year_num"].astype(str) + "-" + 
    (agri_final["year_num"] + 1).astype(str)
)

agri_final.head(15)


,State,District,Year,total_area,total_production,year_num
0,Andaman and Nicobar Islands,Nicobars,2013-2014,13.65,379.70,2013
1,Andaman and Nicobar Islands,Nicobars,2014-2015,26.80,264.80,2014
2,Andaman and Nicobar Islands,Nicobars,2015-2016,11.00,23.20,2015
3,Andaman and Nicobar Islands,Nicobars,2016-2017,25.10,36.54,2016
4,Andaman and Nicobar Islands,Nicobars,2017-2018,18.00,13.73,2017
5,Andaman and Nicobar Islands,Nicobars,2018-2019,7.10,16.01,2018
6,Andaman and Nicobar Islands,Nicobars,2019-2020,6.80,19.30,2019
7,Andaman and Nicobar Islands,Nicobars,2020-2021,3.30,13.50,2020
8,Andaman and Nicobar Islands,Nicobars,2021-2022,5.50,30.10,2021
9,Andaman and Nicobar Islands,Nicobars,2022-2023,1.00,1.26,2022


In [96]:
agri_final["agri_yield_index"] = (
    agri_final["total_production"] /
    agri_final["total_area"]
)

agri_final.head(5)

,State,District,Year,total_area,total_production,year_num,agri_yield_index
0,Andaman and Nicobar Islands,Nicobars,2013-2014,13.65,379.70,2013,27.816850
1,Andaman and Nicobar Islands,Nicobars,2014-2015,26.80,264.80,2014,9.880597
2,Andaman and Nicobar Islands,Nicobars,2015-2016,11.00,23.20,2015,2.109091
3,Andaman and Nicobar Islands,Nicobars,2016-2017,25.10,36.54,2016,1.455777
4,Andaman and Nicobar Islands,Nicobars,2017-2018,18.00,13.73,2017,0.762778


In [97]:
# Drop the old Year column (formatted as "2013-2014") and rename year_num to Year
agri_final = agri_final.drop(columns=["Year"])
agri_final = agri_final.rename(columns={"year_num": "Year"})

agri_final = agri_final[["State", "District", "Year", "total_area", "total_production", "agri_yield_index"]]

agri_final.head(15)


,State,District,Year,total_area,total_production,agri_yield_index
0,Andaman and Nicobar Islands,Nicobars,2013,13.65,379.70,27.816850
1,Andaman and Nicobar Islands,Nicobars,2014,26.80,264.80,9.880597
2,Andaman and Nicobar Islands,Nicobars,2015,11.00,23.20,2.109091
3,Andaman and Nicobar Islands,Nicobars,2016,25.10,36.54,1.455777
4,Andaman and Nicobar Islands,Nicobars,2017,18.00,13.73,0.762778
5,Andaman and Nicobar Islands,Nicobars,2018,7.10,16.01,2.254930
6,Andaman and Nicobar Islands,Nicobars,2019,6.80,19.30,2.838235
7,Andaman and Nicobar Islands,Nicobars,2020,3.30,13.50,4.090909
8,Andaman and Nicobar Islands,Nicobars,2021,5.50,30.10,5.472727
9,Andaman and Nicobar Islands,Nicobars,2022,1.00,1.26,1.260000


In [98]:
agri_final.shape

#unique districts
agri_final[["State","District"]].drop_duplicates().shape

(642, 2)

In [99]:
agri_final.to_csv("district_agriculture_panel.csv", index=False)